# 03 - Entrenamiento de Modelos

Evaluacion de multiples algoritmos con MLflow tracking

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cargar datos procesados
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train = train.drop('Churn', axis=1)
y_train = train['Churn']
X_test = test.drop('Churn', axis=1)
y_test = test['Churn']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Configurar MLflow
# mlflow.set_tracking_uri('https://dagshub.com/YOUR_USER/YOUR_REPO.mlflow')
# mlflow.set_experiment('churn-prediction')

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    print(f'\n--- {model_name} ---')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}')
    print('\n', classification_report(y_test, y_pred))
    
    return metrics, y_proba

In [ ]:
# Modelos a evaluar
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    metrics, y_proba = evaluate_model(model, X_test, y_test, name)
    results[name] = {'metrics': metrics, 'y_proba': y_proba, 'model': model}

In [ ]:
# Comparar modelos
metrics_df = pd.DataFrame({
    name: r['metrics'] for name, r in results.items()
}).T

fig, ax = plt.subplots(figsize=(10, 6))
metrics_df.plot(kind='bar', ax=ax)
plt.title('Comparacion de Modelos')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
plt.figure(figsize=(8, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['metrics']['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Registrar experimentos en MLflow
# with mlflow.start_run(run_name='best-model'):
#     mlflow.log_params(best_model.get_params())
#     mlflow.log_metrics(best_metrics)
#     mlflow.sklearn.log_model(best_model, 'model')